# Hybrid Retrieval Theory: When One Search Signal Lies

**Guiding question:** How can Riverside place the current, authorized policy first when one
employee types `RIGHTS-17` and another asks about "welcoming a baby"?

This is Part **01a** of the RAG sequence. You will make lexical and dense retrieval fail,
watch rank fusion repair complementary errors, and expose the boundaries fusion cannot fix.
Part [01b](01b-hybrid-retrieval-lab.ipynb) resumes the same corpus and builds the complete
PyTorch pipeline.

| Step | Complaint | Evidence |
| --- | --- | --- |
| One retriever | Exact codes and paraphrases pull in opposite directions | Per-query ranks |
| Fusion | Raw scores are incomparable | RRF contribution table and animation |
| Candidate depth | Fusion cannot resurrect an omitted passage | Recall by depth |
| Authorization | A relevant result can still be forbidden | Pre/post-filter comparison |
| Support | A ranker always returns something | Unsupported-query abstention |

**Scope:** retrieval mechanics are built and measured here. Reranking, fallbacks, telemetry,
and manifests are implemented in 01b. Answer-stage diagnosis belongs to
[RAG failure localization](02-rag-failure-localization-and-evaluation.ipynb).


## 0 - The visible failure

Riverside keeps changing policies out of model weights because they need citations,
revocation, revision tracking, and access control. Retrieval now has one job: put a
current, authorized, supporting passage near the top.

**Predict:** Which search method should win on an exact code? Which should win when the
employee never says "parental leave"?

![Static hybrid-retrieval storyboard](images/hybrid-retrieval-storyboard.png)


In [ ]:
# Deterministic setup and shared Riverside fixtures.
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

from rag_shared import (
    COURSE_CAVEAT, DOCUMENTS, RETRIEVAL_QUERIES, SEMANTIC_FEATURES,
    assert_fixture_contracts, set_seed, tokenize,
)

set_seed()
assert_fixture_contracts()
print(COURSE_CAVEAT)
print(f"Corpus: {len(DOCUMENTS)} versioned policy passages")


In [ ]:
# Small transparent lexical and dense scorers for the theory experiment.
doc_tokens = [tokenize(document["text"]) for document in DOCUMENTS]
document_frequency = Counter(token for row in doc_tokens for token in set(row))
idf = {
    token: math.log((len(DOCUMENTS) + 1) / (count + 0.5))
    for token, count in document_frequency.items()
}
feature_names = sorted(set(SEMANTIC_FEATURES.values()))
feature_index = {name: index for index, name in enumerate(feature_names)}

def lexical_score(query, document):
    counts = Counter(tokenize(document["text"]))
    return sum(idf.get(token, 0.0) * min(counts[token], 2) for token in tokenize(query))

def dense_vector(text):
    vector = torch.zeros(len(feature_names), dtype=torch.float32)
    for token in tokenize(text):
        feature = SEMANTIC_FEATURES.get(token)
        if feature:
            vector[feature_index[feature]] += 1
    return torch.nn.functional.normalize(vector, dim=0) if vector.norm() else vector

def rank(query, scorer):
    scored = [(document["id"], float(scorer(query, document))) for document in DOCUMENTS]
    return sorted(scored, key=lambda item: (-item[1], item[0]))


### Attempt 1: trust lexical search

Lexical scoring preserves rare strings. That sounds ideal until the user describes the
policy without its official name.


In [ ]:
# Measure the complementary failures on the two canonical questions.
def dense_score(query, document):
    return torch.dot(dense_vector(query), dense_vector(document["text"])).item()

probes = [RETRIEVAL_QUERIES[0], RETRIEVAL_QUERIES[1]]
rows = []
for probe in probes:
    for method, scorer in [("lexical", lexical_score), ("dense", dense_score)]:
        ranking = rank(probe["question"], scorer)
        gold_rank = next(
            (index for index, (doc_id, _) in enumerate(ranking, start=1)
             if doc_id in probe["gold_ids"]),
            None,
        )
        rows.append({
            "query": probe["kind"],
            "method": method,
            "top_document": ranking[0][0],
            "gold_rank": gold_rank,
        })

failure_table = pd.DataFrame(rows)
display(failure_table)
assert failure_table.query("query == 'exact code' and method == 'lexical'").iloc[0]["gold_rank"] == 1
assert failure_table.query("query == 'paraphrase' and method == 'dense'").iloc[0]["gold_rank"] == 1
print("Complaint: each retriever wins exactly where the other loses.")


### Attempt 2: choose dense search instead

You fixed the paraphrase. You also made the exact identifier nearly meaningless: this toy
semantic encoder intentionally has no learned representation for `RIGHTS-17`.

**What remains:** choosing one retriever moves the blind spot. You need both candidate lists,
but their raw scores use different units.


In [ ]:
# Measure top-3 overlap: agreement is evidence, disagreement is a diagnostic.
overlap_rows = []
for probe in RETRIEVAL_QUERIES[:5]:
    lexical_ids = [doc_id for doc_id, _ in rank(probe["question"], lexical_score)[:3]]
    dense_ids = [doc_id for doc_id, _ in rank(probe["question"], dense_score)[:3]]
    overlap_rows.append({
        "query": probe["id"],
        "overlap_at_3": len(set(lexical_ids) & set(dense_ids)),
        "lexical_top": lexical_ids[0],
        "dense_top": dense_ids[0],
    })
overlap = pd.DataFrame(overlap_rows)
display(overlap)
print("Low overlap is not automatically bad; it shows the retrievers carry different evidence.")


## 1 - RRF: fuse positions, not incompatible scores

Reciprocal Rank Fusion gives each document a contribution of `1 / (k + rank)` from every
list that contains it. The constant `k` controls how sharply early ranks dominate.

**Predict:** Will a passage ranked first by one retriever and second by the other beat a
passage ranked first by only one list?


In [ ]:
# RRF contribution table for the exact-code query.
def rrf(rankings, k=60):
    totals = Counter()
    contributions = []
    for source, ranking in rankings.items():
        for position, (doc_id, _) in enumerate(ranking, start=1):
            credit = 1 / (k + position)
            totals[doc_id] += credit
            contributions.append({
                "source": source, "document": doc_id,
                "rank": position, "credit": credit,
            })
    fused = sorted(totals.items(), key=lambda item: (-item[1], item[0]))
    return fused, pd.DataFrame(contributions)

query = RETRIEVAL_QUERIES[0]["question"]
source_rankings = {
    "lexical": rank(query, lexical_score)[:4],
    "dense": rank(query, dense_score)[:4],
}
fused, contribution_table = rrf(source_rankings, k=60)
display(contribution_table.head(8))
print("Fused top 3:", [doc_id for doc_id, _ in fused[:3]])


In [ ]:
# Static evidence plus a two-frame animation from the measured RRF contributions.
pivot = contribution_table.pivot_table(
    index="document", columns="source", values="credit", aggfunc="sum", fill_value=0
)
ordered = pivot.assign(total=pivot.sum(axis=1)).sort_values("total", ascending=False).head(4)
ax = ordered[["lexical", "dense"]].plot.bar(stacked=True, figsize=(9, 4))
ax.set_title("RRF credit by retriever (static evidence)")
ax.set_ylabel("reciprocal-rank credit")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
def draw(frame):
    ax.clear()
    columns = ["lexical"] if frame == 0 else ["lexical", "dense"]
    ordered[columns].plot.bar(stacked=True, ax=ax, legend=True)
    ax.set_title(f"RRF evidence accumulated: {' + '.join(columns)}")
    ax.set_ylabel("credit")
animation = FuncAnimation(fig, draw, frames=2, interval=900, repeat=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


In [ ]:
# Change one variable: k alters rank decay but not source membership.
k_rows = []
for k_value in [10, 60, 200]:
    fused_k, _ = rrf(source_rankings, k=k_value)
    rights_rank = next(
        index for index, (doc_id, _) in enumerate(fused_k, start=1)
        if doc_id == "policy-rights-17-r3"
    )
    spread = fused_k[0][1] - fused_k[-1][1]
    k_rows.append({"k": k_value, "rights_rank": rights_rank, "score_spread": spread})
display(pd.DataFrame(k_rows))
print("Tune k on labeled questions; 60 is a baseline, not folklore with a lab coat.")


## 2 - Fusion still has a recall ceiling

RRF can reorder only what at least one retriever supplies. Candidate depth therefore sets a
hard ceiling before reranking begins.


In [ ]:
# Show the dense exact-code miss disappearing only when candidate depth grows.
exact_probe = RETRIEVAL_QUERIES[0]
dense_ranking = rank(exact_probe["question"], dense_score)
depth_rows = []
for depth in [1, 2, 3, 6]:
    candidates = {doc_id for doc_id, _ in dense_ranking[:depth]}
    depth_rows.append({
        "candidate_depth": depth,
        "gold_present": bool(candidates & exact_probe["gold_ids"]),
    })
depth_evidence = pd.DataFrame(depth_rows)
display(depth_evidence)
print("If gold_present is false, fusion and reranking have zero chance to recover the passage.")


In [ ]:
# Recall@K and MRR answer different questions.
def recall_at_k(ranking, gold_ids, k):
    return len({doc_id for doc_id, _ in ranking[:k]} & gold_ids) / len(gold_ids)

def reciprocal_rank(ranking, gold_ids):
    return next(
        (1 / rank_index for rank_index, (doc_id, _) in enumerate(ranking, start=1)
         if doc_id in gold_ids),
        0.0,
    )

metric_rows = []
for probe in RETRIEVAL_QUERIES[:5]:
    for method, scorer in [("lexical", lexical_score), ("dense", dense_score)]:
        ranking = rank(probe["question"], scorer)
        metric_rows.append({
            "query": probe["id"], "method": method,
            "recall@3": recall_at_k(ranking, probe["gold_ids"], 3),
            "mrr": reciprocal_rank(ranking, probe["gold_ids"]),
        })
metrics = pd.DataFrame(metric_rows)
display(metrics.groupby("method")[["recall@3", "mrr"]].mean())


## 3 - Relevance is not authorization

A late ACL filter is unsafe: a forbidden passage can influence ranks, logs, snippets, or the
generator before being removed. Authorization must reduce the candidate universe **before**
lexical scoring, dense scoring, fusion, and reranking.


In [ ]:
# Compare unsafe post-filtering with safe pre-filtering for an employee.
employee_groups = {"employee"}
rights_question = RETRIEVAL_QUERIES[0]["question"]

unsafe_scored = rank(rights_question, lexical_score)
unsafe_top_before_filter = unsafe_scored[0][0]
safe_corpus = [
    document for document in DOCUMENTS
    if document["groups"] & employee_groups
]
safe_scored = sorted(
    [(document["id"], lexical_score(rights_question, document)) for document in safe_corpus],
    key=lambda item: (-item[1], item[0]),
)
print("Unsafe top result before late filter:", unsafe_top_before_filter)
print("Safe top result after pre-filter:", safe_scored[0][0])
assert unsafe_top_before_filter == "policy-rights-17-r3"
assert all(doc_id != "policy-rights-17-r3" for doc_id, _ in safe_scored)
print("Authorization uncertainty must fail closed; relevance never broadens access.")


## 4 - A ranker always returns a winner

The unsupported dental-deductible question is topically close to Benefits. That does not
mean the corpus contains the requested deductible. A support gate asks a different question:
does the passage contain enough evidence for the requested claim?


In [ ]:
# Transparent support check: unresolved requested details force abstention.
def support_score(question, document):
    lexical = float(lexical_score(question, document) > 0)
    semantic = max(dense_score(question, document), 0.0)
    requested_details = {"deductible", "dependants"}
    missing_details = requested_details & set(tokenize(question)) - set(tokenize(document["text"]))
    penalty = 0.8 if missing_details else 0.0
    return max(0.0, min(1.0, 0.3 * lexical + 0.7 * semantic - penalty))

unsupported = RETRIEVAL_QUERIES[-1]
ranked = rank(unsupported["question"], lexical_score)
top_document = next(document for document in DOCUMENTS if document["id"] == ranked[0][0])
score = support_score(unsupported["question"], top_document)
decision = "ANSWER" if score >= 0.45 else "ABSTAIN"
print("Topical winner:", top_document["title"])
print(f"Support score: {score:.3f} -> {decision}")
assert decision == "ABSTAIN"


## Scorecard and boundary

| Opening question | Measured answer |
| --- | --- |
| Why keep lexical retrieval? | The exact policy code reached rank 1 lexically. |
| Why keep dense retrieval? | The baby/parental-leave paraphrase reached rank 1 densely. |
| Why RRF? | Contributions used ranks rather than incompatible raw units. |
| What can fusion not fix? | Missing candidates produced a measured zero-recovery ceiling. |
| Where does ACL run? | Before every scoring path; late filtering exposed a forbidden top result. |
| Why a support gate? | The unsupported query still had a topical winner and correctly abstained. |

**Built and measured:** lexical/dense failures, overlap, RRF movement, `k`, candidate depth,
Recall@K/MRR, ACL ordering, and support gating.

**Explained:** weighted fusion and cross-encoder reranking.

**Next:** [01b](01b-hybrid-retrieval-lab.ipynb) builds the PyTorch encoder, weighted and RRF
sweeps, reranker, degraded mode, telemetry, and atomic index manifest. The unresolved
question is operational: can every stage fail without violating authorization or evidence?
